# 🌸 Saffron Detection & Extraction from Weeds/Plants
### Computer Vision Pipeline using HSV Color Segmentation, Morphological Processing & Deep Learning

**Objective:** Detect and isolate *Crocus sativus* (saffron stigmas/flowers) from surrounding weeds and foliage using:
- HSV-based color thresholding
- Morphological operations
- Contour detection & shape analysis
- CNN-based classification (transfer learning)
- Bounding box localization & "plucking" mask generation


## 1. Install & Import Dependencies

In [ ]:
# Install required packages (run once)
# !pip install opencv-python-headless numpy matplotlib scikit-image torch torchvision Pillow scikit-learn seaborn

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import os
import warnings
warnings.filterwarnings('ignore')

# Scikit-image
from skimage import morphology, measure, color, filters
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy import ndimage as ndi

# PyTorch (for CNN classification)
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image

# Sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("✅ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 2. Synthetic Dataset Generator
Since real saffron field images may not be available, we generate realistic synthetic images with:
- **Saffron** (vivid purple/violet flowers with deep red-orange stigmas)
- **Weeds** (green foliage of varying shades)
- **Soil** background texture


In [ ]:
def generate_saffron_field(img_size=512, n_saffron=6, n_weeds=20, seed=42):
    """
    Generate a synthetic saffron field image with ground-truth masks.
    Returns: (bgr_image, gt_mask) where gt_mask=1 for saffron, 0 for background.
    """
    np.random.seed(seed)
    # ----- Soil background -----
    base = np.random.randint(60, 110, (img_size, img_size, 3), dtype=np.uint8)
    base[:, :, 0] = np.clip(base[:, :, 0] + 20, 0, 255)  # brownish

    gt_mask = np.zeros((img_size, img_size), dtype=np.uint8)

    # ----- Weeds (green blobs) -----
    for _ in range(n_weeds):
        cx, cy = np.random.randint(20, img_size-20, 2)
        axes = (np.random.randint(15, 55), np.random.randint(10, 35))
        angle = np.random.randint(0, 180)
        g = np.random.randint(80, 200)
        weed_color = (np.random.randint(10, 60), g, np.random.randint(10, 60))
        cv2.ellipse(base, (cx, cy), axes, angle, 0, 360, weed_color, -1)

    # ----- Saffron flowers -----
    saffron_regions = []
    for _ in range(n_saffron):
        cx, cy = np.random.randint(40, img_size-40, 2)
        # Purple petals
        petal_r = np.random.randint(18, 32)
        for p in range(6):
            ang = p * 60 + np.random.randint(-10, 10)
            px = int(cx + petal_r * np.cos(np.radians(ang)))
            py = int(cy + petal_r * np.sin(np.radians(ang)))
            cv2.ellipse(base, (px, py), (12, 6), ang,
                        0, 360, (np.random.randint(120,160), 30, np.random.randint(160,220)), -1)
        # Red-orange stigma (the saffron part)
        stigma_color_bgr = (10, np.random.randint(60, 110), np.random.randint(180, 230))
        cv2.ellipse(base, (cx, cy), (7, 4), 0, 0, 360, stigma_color_bgr, -1)
        cv2.ellipse(gt_mask, (cx, cy), (9, 6), 0, 0, 360, 1, -1)
        saffron_regions.append((cx, cy))

    # Apply slight Gaussian blur for realism
    base = cv2.GaussianBlur(base, (3, 3), 0.8)
    return base, gt_mask, saffron_regions


# Generate & visualize
img_bgr, gt_mask, saffron_pts = generate_saffron_field(img_size=512, n_saffron=7, n_weeds=25)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_rgb); axes[0].set_title("Synthetic Saffron Field", fontsize=14, fontweight='bold')
axes[0].axis('off')
axes[1].imshow(gt_mask, cmap='hot'); axes[1].set_title("Ground-Truth Saffron Mask", fontsize=14, fontweight='bold')
axes[1].axis('off')
plt.suptitle("Step 1: Input Image + Ground Truth", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/step1_input.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Image shape: {img_bgr.shape}  |  Saffron pixels: {gt_mask.sum()}")


## 3. Preprocessing Pipeline
Key steps:
- Convert BGR → HSV (Hue-Saturation-Value) for robust color separation
- Apply **CLAHE** (Contrast Limited Adaptive Histogram Equalization) for uneven lighting
- Noise reduction via **bilateral filtering** (preserves edges)


In [ ]:
def preprocess_image(bgr_img):
    """Full preprocessing pipeline."""
    # Bilateral filter: smooths noise while keeping edges sharp
    filtered = cv2.bilateralFilter(bgr_img, d=9, sigmaColor=75, sigmaSpace=75)

    # CLAHE on L-channel of LAB
    lab = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    # Convert to HSV
    hsv = cv2.cvtColor(enhanced, cv2.COLOR_BGR2HSV)
    return filtered, enhanced, hsv


filtered, enhanced, hsv = preprocess_image(img_bgr)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
titles = ['Original', 'Bilateral Filtered', 'CLAHE Enhanced', 'HSV Space']
imgs = [img_rgb,
        cv2.cvtColor(filtered, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB),
        hsv]
cmaps = [None, None, None, None]

for ax, im, t in zip(axes, imgs, titles):
    ax.imshow(im); ax.set_title(t, fontweight='bold'); ax.axis('off')

plt.suptitle("Step 2: Preprocessing Pipeline", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/step2_preprocess.png', dpi=120, bbox_inches='tight')
plt.show()

# HSV channel breakdown
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, name) in enumerate(zip(axes, ['Hue', 'Saturation', 'Value'])):
    ax.imshow(hsv[:, :, i], cmap='plasma')
    ax.set_title(f'{name} Channel', fontweight='bold'); ax.axis('off')
plt.tight_layout(); plt.show()


## 4. Multi-Stage HSV Color Segmentation

### Saffron Color Ranges in HSV:
| Component | HSV Range |
|-----------|-----------|
| **Stigma (red-orange)** | H: 0–20 & 160–180, S: 100–255, V: 100–255 |
| **Petal (purple/violet)** | H: 120–160, S: 60–255, V: 60–255 |
| **Weed (green)** | H: 35–85, S: 40–255, V: 40–255 |


In [ ]:
def segment_saffron_hsv(hsv_img):
    """
    Multi-range HSV thresholding to capture saffron stigma (red-orange)
    and flower petals (purple/violet).
    """
    # --- Saffron stigma: red-orange ---
    lower_red1 = np.array([0,   100, 100])
    upper_red1 = np.array([18,  255, 255])
    lower_red2 = np.array([160, 100, 100])
    upper_red2 = np.array([180, 255, 255])
    mask_red1 = cv2.inRange(hsv_img, lower_red1, upper_red1)
    mask_red2 = cv2.inRange(hsv_img, lower_red2, upper_red2)
    mask_stigma = cv2.bitwise_or(mask_red1, mask_red2)

    # --- Saffron flower petals: purple/violet ---
    lower_purple = np.array([120, 50, 50])
    upper_purple = np.array([160, 255, 255])
    mask_petal = cv2.inRange(hsv_img, lower_purple, upper_purple)

    # Combined saffron mask
    mask_saffron_raw = cv2.bitwise_or(mask_stigma, mask_petal)

    # --- Weed: green ---
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    mask_weed = cv2.inRange(hsv_img, lower_green, upper_green)

    return mask_saffron_raw, mask_stigma, mask_petal, mask_weed


mask_saffron_raw, mask_stigma, mask_petal, mask_weed = segment_saffron_hsv(hsv)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
data = [mask_stigma, mask_petal, mask_saffron_raw, mask_weed]
titles = ['Stigma Mask
(Red-Orange)', 'Petal Mask
(Purple)', 'Combined Saffron
Mask (Raw)', 'Weed Mask
(Green)']
cmaps_list = ['Oranges', 'Purples', 'YlOrRd', 'Greens']

for ax, d, t, c in zip(axes, data, titles, cmaps_list):
    ax.imshow(d, cmap=c); ax.set_title(t, fontweight='bold', fontsize=11); ax.axis('off')

plt.suptitle("Step 3: HSV Multi-Range Color Segmentation", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/step3_segmentation.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Stigma pixels detected:  {mask_stigma.sum()//255:>6}")
print(f"Petal pixels detected:   {mask_petal.sum()//255:>6}")
print(f"Saffron total:           {mask_saffron_raw.sum()//255:>6}")
print(f"Weed pixels detected:    {mask_weed.sum()//255:>6}")


## 5. Morphological Operations & Noise Removal
- **Opening** (erosion → dilation): removes small false-positive speckles
- **Closing** (dilation → erosion): fills small holes inside saffron blobs
- **Hole filling** via `ndimage.binary_fill_holes`


In [ ]:
def morphological_refinement(mask, open_ksize=5, close_ksize=9, min_area=80):
    """Clean up a binary mask with morphological ops + area filtering."""
    kernel_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_ksize,  open_ksize))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ksize, close_ksize))

    # Remove small noise
    opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel_open,  iterations=2)
    # Fill gaps
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel_close, iterations=2)

    # Fill holes
    filled = ndi.binary_fill_holes(closed).astype(np.uint8) * 255

    # Remove tiny blobs by area
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(filled, connectivity=8)
    clean = np.zeros_like(filled)
    for lbl in range(1, num_labels):
        if stats[lbl, cv2.CC_STAT_AREA] >= min_area:
            clean[labels == lbl] = 255

    return clean


mask_saffron_clean = morphological_refinement(mask_saffron_raw, open_ksize=3, close_ksize=7, min_area=60)
mask_stigma_clean  = morphological_refinement(mask_stigma,      open_ksize=3, close_ksize=5, min_area=30)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(mask_saffron_raw,   cmap='hot'); axes[0].set_title('Raw Mask',          fontweight='bold'); axes[0].axis('off')
axes[1].imshow(mask_saffron_clean, cmap='hot'); axes[1].set_title('After Morphology',  fontweight='bold'); axes[1].axis('off')

# Overlay on original
overlay = img_rgb.copy()
overlay[mask_saffron_clean > 0] = (255, 50, 50)
axes[2].imshow(overlay); axes[2].set_title('Saffron Overlay',  fontweight='bold'); axes[2].axis('off')

plt.suptitle("Step 4: Morphological Refinement", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/step4_morph.png', dpi=120, bbox_inches='tight')
plt.show()


## 6. Contour Detection & Shape-Based Filtering
Saffron stigmas have characteristic shape metrics:
- **Circularity** = 4π·Area / Perimeter² ≈ 0.4–1.0 (elongated thread-like)
- **Aspect Ratio** of bounding box ≈ 1.5–4.0
- **Area** filter to discard outliers


In [ ]:
def extract_saffron_contours(clean_mask, img_rgb,
                              min_area=40, max_area=8000,
                              min_circ=0.25, min_ar=1.0, max_ar=6.0):
    """
    Find contours, compute shape descriptors, filter for saffron-like shapes.
    Returns list of dicts with bbox, mask, and shape stats.
    """
    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    detections = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area or area > max_area:
            continue

        perimeter = cv2.arcLength(cnt, True)
        circularity = (4 * np.pi * area / (perimeter ** 2 + 1e-5))

        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = max(w, h) / (min(w, h) + 1e-5)

        if circularity < min_circ:
            continue

        # Solidity
        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull) + 1e-5
        solidity = area / hull_area

        detections.append({
            'contour':       cnt,
            'bbox':          (x, y, w, h),
            'area':          area,
            'circularity':   round(circularity, 3),
            'aspect_ratio':  round(aspect_ratio, 3),
            'solidity':      round(solidity, 3),
        })

    return detections


detections = extract_saffron_contours(mask_saffron_clean, img_rgb)

# Visualize detections
canvas = img_rgb.copy()
for i, det in enumerate(detections):
    x, y, w, h = det['bbox']
    cv2.rectangle(canvas, (x, y), (x+w, y+h), (255, 255, 0), 2)
    cv2.drawContours(canvas, [det['contour']], -1, (255, 50, 50), 2)
    cv2.putText(canvas, f"S{i+1}", (x, y-4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,0), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_rgb);  axes[0].set_title('Original',             fontweight='bold'); axes[0].axis('off')
axes[1].imshow(canvas);   axes[1].set_title(f'Detections ({len(detections)} saffron regions)', fontweight='bold'); axes[1].axis('off')

plt.suptitle("Step 5: Contour Detection & Shape Filtering", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/step5_contours.png', dpi=120, bbox_inches='tight')
plt.show()

# Print shape stats
print(f"{'ID':>3} {'Area':>7} {'Circularity':>12} {'Aspect Ratio':>13} {'Solidity':>9}")
print("-" * 50)
for i, det in enumerate(detections):
    print(f"S{i+1:>2} {det['area']:>7.0f} {det['circularity']:>12.3f} {det['aspect_ratio']:>13.3f} {det['solidity']:>9.3f}")


## 7. Watershed Segmentation for Touching Regions
When saffron flowers/stigmas overlap, watershed separates them by treating pixel intensities as a topographic map.


In [ ]:
def watershed_segmentation(bgr_img, binary_mask):
    """Apply marker-based watershed to separate touching saffron blobs."""
    dist_transform = ndi.distance_transform_edt(binary_mask > 0)

    # Local maxima as seeds
    local_max_coords = peak_local_max(dist_transform, min_distance=12,
                                       labels=binary_mask > 0)
    local_max_mask = np.zeros(dist_transform.shape, dtype=bool)
    local_max_mask[tuple(local_max_coords.T)] = True

    markers, n_markers = ndi.label(local_max_mask)
    labels = watershed(-dist_transform, markers, mask=binary_mask > 0)

    return labels, n_markers


labels_ws, n_ws = watershed_segmentation(img_bgr, mask_saffron_clean)

# Color-code each region
label_color = color.label2rgb(labels_ws, img_rgb, kind='overlay', alpha=0.5)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_rgb);                       axes[0].set_title('Original',              fontweight='bold'); axes[0].axis('off')
axes[1].imshow(mask_saffron_clean, cmap='hot'); axes[1].set_title('Clean Mask',            fontweight='bold'); axes[1].axis('off')
axes[2].imshow(label_color);                   axes[2].set_title(f'Watershed — {n_ws} regions', fontweight='bold'); axes[2].axis('off')

plt.suptitle("Step 6: Watershed Segmentation (Separating Touching Regions)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/step6_watershed.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Watershed separated {n_ws} individual saffron regions.")


## 8. CNN-Based Patch Classifier (Transfer Learning)
Each detected region is cropped into a patch and classified by a **MobileNetV2** backbone (pretrained on ImageNet) with a custom head:
- `saffron` vs `non-saffron`

In production, fine-tune on a labelled saffron dataset. Here we demonstrate the architecture and inference pipeline.


In [ ]:
class SaffronClassifier(nn.Module):
    """MobileNetV2 backbone + binary classification head."""
    def __init__(self, pretrained=True):
        super().__init__()
        base = models.mobilenet_v2(weights='IMAGENET1K_V1' if pretrained else None)
        # Freeze backbone
        for p in base.features.parameters():
            p.requires_grad = False
        in_features = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)   # [non-saffron, saffron]
        )
        self.model = base

    def forward(self, x):
        return self.model(x)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clf_model = SaffronClassifier(pretrained=True).to(device)
clf_model.eval()

# Inference transform
transform_infer = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

def classify_patch(bgr_patch, model, transform, device):
    """Return (label, confidence) for a BGR crop."""
    rgb_patch = cv2.cvtColor(bgr_patch, cv2.COLOR_BGR2RGB)
    pil_img   = Image.fromarray(rgb_patch)
    tensor    = transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]
    label = 'saffron' if probs[1] > 0.5 else 'non-saffron'
    return label, float(probs[1])


# Classify all detected regions
print(f"{'ID':>4}  {'Label':>12}  {'Saffron Conf':>13}  {'Area':>7}  {'Circularity':>12}")
print("-" * 56)
classified = []
for i, det in enumerate(detections):
    x, y, w, h = det['bbox']
    pad = 6
    x1, y1 = max(0, x-pad), max(0, y-pad)
    x2, y2 = min(img_bgr.shape[1], x+w+pad), min(img_bgr.shape[0], y+h+pad)
    patch   = img_bgr[y1:y2, x1:x2]
    if patch.size == 0:
        continue
    label, conf = classify_patch(patch, clf_model, transform_infer, device)
    classified.append({**det, 'label': label, 'conf': conf})
    print(f"S{i+1:>3}  {label:>12}  {conf:>12.3f}  {det['area']:>7.0f}  {det['circularity']:>12.3f}")

print(f"\n✅ Confirmed saffron regions: {sum(1 for c in classified if c['label']=='saffron')}")


## 9. "Plucking" Mask Generation
Generate the final **actionable output**:
- **Red highlight** = confirmed saffron stigma to be plucked
- **Yellow bounding box** = robotic arm targeting region
- **Green** = weeds/other plants (to be ignored)


In [ ]:
def generate_plucking_output(img_rgb, classified, mask_weed, mask_saffron_clean):
    """
    Produce:
      1. Annotated image with pluck targets
      2. Binary pluck mask (255 = pluck here)
      3. Weed avoidance overlay
    """
    annotated = img_rgb.copy()
    pluck_mask = np.zeros(img_rgb.shape[:2], dtype=np.uint8)

    # Shade weeds green
    weed_overlay = annotated.copy()
    weed_overlay[mask_weed > 0] = (30, 160, 30)
    annotated = cv2.addWeighted(annotated, 0.7, weed_overlay, 0.3, 0)

    for det in classified:
        x, y, w, h = det['bbox']
        if det['label'] == 'saffron':
            # Mark pluck region
            cv2.drawContours(pluck_mask, [det['contour']], -1, 255, -1)
            cv2.rectangle(annotated, (x-2, y-2), (x+w+2, y+h+2), (255, 220, 0), 2)
            cv2.drawContours(annotated, [det['contour']], -1, (255, 50, 30), 2)
            label_txt = f"PLUCK {det['conf']:.0%}"
            cv2.rectangle(annotated, (x, y-18), (x + len(label_txt)*8, y), (255, 220, 0), -1)
            cv2.putText(annotated, label_txt, (x+2, y-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (20, 20, 20), 1)
        else:
            cv2.rectangle(annotated, (x, y), (x+w, y+h), (100, 100, 100), 1)

    return annotated, pluck_mask


annotated, pluck_mask = generate_plucking_output(img_rgb, classified, mask_weed, mask_saffron_clean)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
axes[0].imshow(img_rgb);    axes[0].set_title('Original Field Image', fontweight='bold', fontsize=13); axes[0].axis('off')
axes[1].imshow(annotated);  axes[1].set_title('🌸 Plucking Target Map', fontweight='bold', fontsize=13); axes[1].axis('off')
axes[2].imshow(pluck_mask, cmap='hot'); axes[2].set_title('Binary Pluck Mask', fontweight='bold', fontsize=13); axes[2].axis('off')

# Legend
legend_elements = [
    mpatches.Patch(facecolor=(1,.87,0), label='Saffron — PLUCK'),
    mpatches.Patch(facecolor=(0,.6,0),  label='Weed — AVOID'),
    mpatches.Patch(facecolor='gray',    label='Other — IGNORE'),
]
axes[1].legend(handles=legend_elements, loc='lower left', fontsize=9,
               framealpha=0.85, facecolor='white')

plt.suptitle("Step 7: Final Plucking Decision Map", fontsize=17, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/step7_plucking.png', dpi=140, bbox_inches='tight')
plt.show()

n_pluck = sum(1 for c in classified if c['label'] == 'saffron')
print(f"🌸 Total saffron targets identified for plucking: {n_pluck}")
print(f"🌿 Weed pixels to avoid:                         {(mask_weed > 0).sum()}")
print(f"📍 Pluck mask coverage (pixels):                  {(pluck_mask > 0).sum()}")


## 10. Evaluation Metrics (vs Ground Truth)
Quantify detection performance using standard segmentation metrics:
- **IoU** (Intersection over Union)
- **Dice Coefficient** (F1 for segmentation)
- **Precision & Recall**


In [ ]:
def evaluate_segmentation(pred_mask, gt_mask, threshold=127):
    """Compute IoU, Dice, Precision, Recall."""
    pred  = (pred_mask  > threshold).astype(bool)
    truth = (gt_mask    > 0        ).astype(bool)

    tp = (pred & truth ).sum()
    fp = (pred & ~truth).sum()
    fn = (~pred & truth).sum()
    tn = (~pred & ~truth).sum()

    iou       = tp / (tp + fp + fn + 1e-7)
    dice      = 2*tp / (2*tp + fp + fn + 1e-7)
    precision = tp / (tp + fp + 1e-7)
    recall    = tp / (tp + fn + 1e-7)
    accuracy  = (tp + tn) / (tp + tn + fp + fn + 1e-7)

    return dict(IoU=iou, Dice=dice, Precision=precision, Recall=recall, Accuracy=accuracy)


metrics = evaluate_segmentation(pluck_mask, gt_mask)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(metrics.keys())
vals  = [metrics[k] for k in names]
colors_bar = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']

bars = axes[0].bar(names, vals, color=colors_bar, edgecolor='white', linewidth=1.5)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Segmentation Metrics', fontweight='bold', fontsize=13)
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].axhline(0.7, color='gray', linestyle='--', alpha=0.5, label='0.7 threshold')
axes[0].legend()

# Confusion overlay
comp = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
pred_b  = (pluck_mask > 127)
truth_b = (gt_mask    > 0  )
comp[pred_b  &  truth_b] = [0,   200, 0]    # TP — green
comp[pred_b  & ~truth_b] = [200, 0,   0]    # FP — red
comp[~pred_b &  truth_b] = [0,   0,   200]  # FN — blue
axes[1].imshow(img_rgb, alpha=0.6)
axes[1].imshow(comp, alpha=0.5)
axes[1].set_title('TP/FP/FN Map (Green/Red/Blue)', fontweight='bold', fontsize=13)
axes[1].axis('off')

leg = [mpatches.Patch(color=(0,.78,0), label='True Positive'),
       mpatches.Patch(color=(0.78,0,0), label='False Positive'),
       mpatches.Patch(color=(0,0,.78),  label='False Negative')]
axes[1].legend(handles=leg, loc='lower right', fontsize=9)

plt.suptitle("Step 8: Quantitative Evaluation", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/step8_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n📊 Segmentation Performance:")
for k, v in metrics.items():
    bar = '█' * int(v * 20)
    print(f"  {k:<12}: {v:.4f}  {bar}")


## 11. Batch Processing Pipeline
Run the full pipeline on multiple images and aggregate results — ready for integration with a robotic harvester or drone.


In [ ]:
def full_pipeline(bgr_img):
    """End-to-end pipeline: image → pluck coordinates."""
    _, enhanced, hsv        = preprocess_image(bgr_img)
    mask_raw, _, _, mask_w  = segment_saffron_hsv(hsv)
    mask_clean              = morphological_refinement(mask_raw)
    detections              = extract_saffron_contours(mask_clean, bgr_img)
    results = []
    for det in detections:
        x, y, w, h = det['bbox']
        pad = 6
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(bgr_img.shape[1], x+w+pad), min(bgr_img.shape[0], y+h+pad)
        patch = bgr_img[y1:y2, x1:x2]
        if patch.size == 0:
            continue
        label, conf = classify_patch(patch, clf_model, transform_infer, device)
        if label == 'saffron':
            cx, cy = x + w//2, y + h//2
            results.append({'center': (cx, cy), 'bbox': (x,y,w,h),
                             'conf': conf, 'area': det['area']})
    return results, mask_w


# Batch: 5 different synthetic scenes
print("Batch Processing Results")
print("=" * 55)
all_counts = []
for seed in range(5):
    img_b, gt_b, _ = generate_saffron_field(seed=seed*13+7,
                                             n_saffron=np.random.randint(4,10),
                                             n_weeds=np.random.randint(15,30))
    res, _ = full_pipeline(img_b)
    all_counts.append(len(res))
    coords  = [r['center'] for r in res]
    confs   = [r['conf']   for r in res]
    print(f"  Scene {seed+1}: {len(res):>2} saffron detected | "
          f"avg conf {np.mean(confs):.2f}" if confs else f"  Scene {seed+1}:  0 saffron detected")

# Summary bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([f'Scene {i+1}' for i in range(5)], all_counts,
       color='#e74c3c', edgecolor='white', linewidth=1.5)
ax.set_ylabel('Saffron Regions Detected', fontsize=12)
ax.set_title('Batch Detection Summary', fontweight='bold', fontsize=14)
for i, v in enumerate(all_counts):
    ax.text(i, v + 0.1, str(v), ha='center', fontweight='bold', fontsize=13)
ax.set_ylim(0, max(all_counts) + 2)
plt.tight_layout()
plt.savefig('/tmp/step9_batch.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"\nMean detections per scene: {np.mean(all_counts):.1f}")


## 12. Pipeline Summary & Next Steps

### ✅ Techniques Applied

| Stage | Technique | Purpose |
|-------|-----------|---------|
| Preprocessing | Bilateral Filter + CLAHE | Noise reduction, lighting correction |
| Color Seg. | Multi-range HSV Thresholding | Isolate saffron red-orange & purple |
| Morphology | Opening / Closing / Hole Fill | Remove noise, fill gaps |
| Shape Analysis | Circularity, Aspect Ratio, Solidity | Filter non-saffron blobs |
| Segmentation | Watershed + Distance Transform | Separate touching regions |
| Classification | MobileNetV2 (Transfer Learning) | Confirm saffron vs weed patches |
| Evaluation | IoU, Dice, Precision, Recall | Quantitative performance |
| Output | Plucking Mask + Bounding Boxes | Robotic arm targeting |

### 🚀 Production Enhancements
1. **Fine-tune MobileNetV2** on a labelled saffron dataset (Kaggle / field collection)
2. **Replace with YOLOv8** for real-time detection at 30+ FPS
3. **Depth camera integration** (Intel RealSense) for 3D plucking coordinates
4. **Drone imagery pipeline** with geo-referenced pluck maps
5. **Active Learning loop**: flag uncertain detections for human review


In [ ]:
print("🌸 Saffron Detection & Plucking Pipeline — Complete!")
print()
print("Saved intermediate visualizations:")
import glob
for f in sorted(glob.glob('/tmp/step*.png')):
    size_kb = os.path.getsize(f) // 1024
    print(f"  {f}  ({size_kb} KB)")
